# Session 5 · Real-World AI Pipelines and Reinforcement Learning

Generative AI › Track 4 · AI Architecture and Agents

Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Runtime ▸ Run all in Colab). Outputs below were produced by actually executing this notebook.

### Example · `session-05_drift_monitor.py`

In [1]:
"""
Session 5 · Example 1 — Monitoring a live AI pipeline for data drift with the Population Stability Index (PSI).

After deployment, the world changes: prices rise, new customer groups arrive.
PSI compares the distribution of a feature in live traffic with training data:
    PSI = sum over bins of (live% - train%) * ln(live% / train%)
Rule of thumb: < 0.10 stable · 0.10-0.25 moderate shift (watch) · > 0.25 major shift (retrain)

Run:  python session-05_drift_monitor.py
"""
import numpy as np

rng = np.random.default_rng(21)


def psi(train, live, bins=10):
    edges = np.quantile(train, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    t = np.histogram(train, edges)[0] / len(train)
    l = np.histogram(live, edges)[0] / len(live)
    t, l = np.clip(t, 1e-4, None), np.clip(l, 1e-4, None)      # avoid log(0)
    return float(np.sum((l - t) * np.log(l / t)))


def verdict(v):
    return "stable" if v < 0.10 else ("WATCH" if v < 0.25 else "RETRAIN")


# Training data for a food-delivery ETA model
train = {
    "order_value_rs": rng.normal(1500, 400, 20000),
    "distance_km": rng.gamma(2.0, 2.0, 20000),
    "hour_of_day": rng.integers(10, 24, 20000).astype(float),
}

# Four weeks of live traffic: inflation slowly raises order values; a new
# suburb opens in week 3 and lengthens distances.
weeks = {
    "week 1": dict(order_value_rs=(1500, 400), distance_shape=2.0, hours=(10, 24)),
    "week 2": dict(order_value_rs=(1560, 410), distance_shape=2.0, hours=(10, 24)),
    "week 3": dict(order_value_rs=(1650, 420), distance_shape=2.6, hours=(10, 24)),
    "week 4": dict(order_value_rs=(1800, 450), distance_shape=3.2, hours=(8, 24)),
}
print(f"{'':8s} {'order_value_rs':>16s} {'distance_km':>14s} {'hour_of_day':>14s}")
for wk, cfg in weeks.items():
    live = {
        "order_value_rs": rng.normal(*cfg["order_value_rs"], 3000),
        "distance_km": rng.gamma(cfg["distance_shape"], 2.0, 3000),
        "hour_of_day": rng.integers(*cfg["hours"], 3000).astype(float),
    }
    cells = []
    for f in train:
        v = psi(train[f], live[f])
        cells.append(f"{v:.3f} {verdict(v):>7s}")
    print(f"{wk:8s} " + " ".join(f"{c:>15s}" for c in cells))

print("\nAction: trigger the retraining pipeline (Session 4) when any feature shows RETRAIN.")

           order_value_rs    distance_km    hour_of_day
week 1     0.003  stable   0.003  stable   0.004  stable
week 2     0.017  stable   0.002  stable   0.002  stable
week 3     0.146   WATCH   0.209   WATCH   0.003  stable
week 4     0.483 RETRAIN   0.651 RETRAIN   0.120   WATCH

Action: trigger the retraining pipeline (Session 4) when any feature shows RETRAIN.


### Example · `session-05_policy_gradient_tf.py`

In [2]:
"""
Session 5 · Example 3 — Reinforcement learning with TensorFlow: a policy-gradient (REINFORCE) agent.

Problem: an online shop can show one of 4 discount banners. Each banner has an
unknown chance of a purchase (the reward). The agent's policy is a tiny
TensorFlow network (softmax over banners). REINFORCE increases the
log-probability of actions in proportion to (reward - baseline).

Run:  python session-05_policy_gradient_tf.py
"""
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np
import tensorflow as tf

tf.random.set_seed(4)
rng = np.random.default_rng(4)
BANNERS = ["5% off", "free delivery", "buy 1 get 1", "10% off"]
TRUE_RATE = np.array([0.04, 0.09, 0.06, 0.07])      # hidden from the agent

logits = tf.Variable(tf.zeros(len(BANNERS)))        # the policy's parameters
opt = tf.keras.optimizers.Adam(0.05)
baseline = 0.0

for step in range(1, 3001):
    probs = tf.nn.softmax(logits)
    a = int(rng.choice(len(BANNERS), p=probs.numpy()))
    r = float(rng.random() < TRUE_RATE[a])           # 1 = purchase
    baseline = 0.99 * baseline + 0.01 * r            # running average reward
    with tf.GradientTape() as tape:
        logp = tf.math.log(tf.nn.softmax(logits)[a])
        loss = -logp * (r - baseline)                # REINFORCE objective
    opt.apply_gradients([(tape.gradient(loss, logits), logits)])
    if step in (1, 500, 1500, 3000):
        p = tf.nn.softmax(logits).numpy()
        print(f"step {step:4d}  policy: " + "  ".join(f"{b}={x:.2f}" for b, x in zip(BANNERS, p)))

best = BANNERS[int(np.argmax(tf.nn.softmax(logits).numpy()))]
print(f"\nAgent prefers: {best} | truly best: {BANNERS[int(TRUE_RATE.argmax())]}")
print(f"Expected purchase rate of learned policy: {float(tf.nn.softmax(logits).numpy() @ TRUE_RATE):.3f} "
      f"(random policy: {TRUE_RATE.mean():.3f})")

step    1  policy: 5% off=0.25  free delivery=0.25  buy 1 get 1=0.25  10% off=0.25


step  500  policy: 5% off=0.02  free delivery=0.46  buy 1 get 1=0.25  10% off=0.28


step 1500  policy: 5% off=0.01  free delivery=0.89  buy 1 get 1=0.02  10% off=0.08


step 3000  policy: 5% off=0.00  free delivery=0.99  buy 1 get 1=0.00  10% off=0.00

Agent prefers: free delivery | truly best: free delivery
Expected purchase rate of learned policy: 0.090 (random policy: 0.065)


### Example · `session-05_quantum_toy.py`

In [3]:
"""
Session 5 · Example 4 — What TensorFlow Quantum does, simulated with NumPy: a one-qubit variational classifier.

A quantum ML model is a circuit with tunable rotation angles (parameters).
Data is encoded as a rotation, a trainable rotation follows, and we measure
the expectation <Z> in [-1, 1]. A classical optimiser tunes the angle using the
parameter-shift rule:  d<Z>/dθ = (<Z>(θ + π/2) - <Z>(θ - π/2)) / 2
TensorFlow Quantum wraps exactly this loop (circuits from Cirq as Keras layers).

Run:  python session-05_quantum_toy.py
"""
import numpy as np

rng = np.random.default_rng(8)


def ry(angle):
    """Single-qubit rotation about the Y axis."""
    c, s = np.cos(angle / 2), np.sin(angle / 2)
    return np.array([[c, -s], [s, c]])


def expect_z(x, theta):
    state = ry(theta) @ ry(x) @ np.array([1.0, 0.0])     # start in |0>, encode x, apply θ
    return float(state[0] ** 2 - state[1] ** 2)          # <Z> = P(0) - P(1)


# Data: class +1 near angle 0.3, class -1 near angle 2.6 (e.g. two sensor states)
x = np.concatenate([rng.normal(0.3, 0.25, 40), rng.normal(2.6, 0.25, 40)])
y = np.concatenate([np.ones(40), -np.ones(40)])
# Shift the data so the untrained circuit is poor: the classifier must learn θ
x = x + 1.4

theta, lr = 0.0, 0.3
for epoch in range(31):
    preds = np.array([expect_z(xi, theta) for xi in x])
    loss = np.mean((preds - y) ** 2)
    grad = np.mean([2 * (expect_z(xi, theta) - yi) *
                    (expect_z(xi, theta + np.pi / 2) - expect_z(xi, theta - np.pi / 2)) / 2
                    for xi, yi in zip(x, y)])
    if epoch % 10 == 0:
        acc = np.mean(np.sign(preds) == y)
        print(f"epoch {epoch:2d}  theta={theta:+.3f}  loss={loss:.3f}  accuracy={acc:.2f}")
    theta -= lr * grad

print(f"\nLearned rotation θ = {theta:.3f} rad (ideal ≈ -1.40, undoing the shift)")
print("Simulating n qubits needs 2^n amplitudes:", {n: 2 ** n for n in (1, 10, 30, 50)})

epoch  0  theta=+0.000  loss=0.725  accuracy=0.69
epoch 10  theta=-1.160  loss=0.026  accuracy=1.00
epoch 20  theta=-1.267  loss=0.022  accuracy=1.00
epoch 30  theta=-1.284  loss=0.022  accuracy=1.00

Learned rotation θ = -1.284 rad (ideal ≈ -1.40, undoing the shift)
Simulating n qubits needs 2^n amplitudes: {1: 2, 10: 1024, 30: 1073741824, 50: 1125899906842624}


### Example · `session-05_project_delivery_agent.py`

In [4]:
"""
Session 5 · Mini project (full solution) — Q-learning delivery robot in a warehouse.

Real-life problem: a warehouse robot must carry parcels from the dock (S) to the
packing desk (G) while avoiding shelves (#) and a wet-floor zone (~) that costs
extra battery. Nobody writes the route: the robot LEARNS it by trial and error.

Q-learning update:  Q[s,a] += α * (r + γ * max_a' Q[s',a'] - Q[s,a])
Rewards: -1 per move, -15 per wet cell, -5 for bumping a shelf, +100 at the desk.
Exploration: ε-greedy, ε decays from 1.0 to 0.05 over 1,000 episodes.

Run:  python session-05_project_delivery_agent.py
Creates: session-05_q_learning_rewards.png
"""
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

GRID = [
    "S..#....",
    ".#.#.##.",
    ".#...#..",
    ".##~~#.#",
    "...~~...",
    "#.##.##.",
    "......#G",
]
H, W = len(GRID), len(GRID[0])
ACTIONS = {0: (-1, 0, "↑"), 1: (1, 0, "↓"), 2: (0, -1, "←"), 3: (0, 1, "→")}
start = next((r, c) for r in range(H) for c in range(W) if GRID[r][c] == "S")
goal = next((r, c) for r in range(H) for c in range(W) if GRID[r][c] == "G")


def step(state, a):
    dr, dc, _ = ACTIONS[a]
    r, c = state[0] + dr, state[1] + dc
    if not (0 <= r < H and 0 <= c < W) or GRID[r][c] == "#":
        return state, -5, False                  # bumped a wall/shelf
    if (r, c) == goal:
        return (r, c), 100, True
    return (r, c), (-15 if GRID[r][c] == "~" else -1), False   # each move costs battery


rng = np.random.default_rng(0)
Q = np.zeros((H, W, 4))
alpha, gamma, episodes = 0.3, 0.99, 2000
rewards = []
for ep in range(episodes):
    eps = max(0.05, 1.0 - ep / 1000)
    s, total = start, 0
    for _ in range(200):
        a = int(rng.integers(4)) if rng.random() < eps else int(Q[s].argmax())
        s2, r, done = step(s, a)
        Q[s][a] += alpha * (r + gamma * Q[s2].max() * (not done) - Q[s][a])
        s, total = s2, total + r
        if done:
            break
    rewards.append(total)

# Follow the learned greedy policy
s, path = start, [start]
while s != goal and len(path) < 60:
    s, _, _ = step(s, int(Q[s].argmax())); path.append(s)

print(f"Average reward, first 50 episodes: {np.mean(rewards[:50]):8.1f}")
print(f"Average reward, last 50 episodes : {np.mean(rewards[-50:]):8.1f}")
print(f"Learned route: {len(path) - 1} moves, reaches goal: {s == goal}, "
      f"wet cells crossed: {sum(GRID[r][c] == '~' for r, c in path)}")

print("\nGreedy policy (arrows) and route (*):")
for r in range(H):
    row = ""
    for c in range(W):
        ch = GRID[r][c]
        if ch in "#SG~" and (r, c) not in path[1:-1]:
            row += ch
        elif (r, c) in path[1:-1]:
            row += "*"
        else:
            row += ACTIONS[int(Q[r, c].argmax())][2]
    print("   " + " ".join(row))

smooth = np.convolve(rewards, np.ones(25) / 25, mode="valid")
plt.figure(figsize=(7.5, 4))
plt.plot(rewards, color="#A78BFA", alpha=.35, label="episode reward")
plt.plot(range(24, episodes), smooth, color="#FF6B1A", lw=2.5, label="25-episode average")
plt.xlabel("Episode"); plt.ylabel("Total reward"); plt.ylim(-400, 120)
plt.title("Q-learning: the robot discovers a short, dry route"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig("session-05_q_learning_rewards.png", dpi=110)
print("\nSaved chart: session-05_q_learning_rewards.png")

Average reward, first 50 episodes:   -600.7


Average reward, last 50 episodes :     78.4
Learned route: 19 moves, reaches goal: True, wet cells crossed: 0

Greedy policy (arrows) and route (*):
   S * * # * * * *
   ↑ # * # * # # *
   ↑ # * * * # * *
   ↓ # # ~ ~ # * #
   → ↓ ← ~ ~ → * *
   # ↓ # # ↑ # # *
   → → → → ↑ ← # G

Saved chart: session-05_q_learning_rewards.png
